# Data Minimization / GDPR Experiment — Google Colab

Notebook này chạy toàn bộ thí nghiệm `1d / 3d / 5d / all` cho **BPR** và **Content-based**.

## Cách chuẩn bị

Upload một trong hai dạng sau lên Colab:

- `/content/recsys-hust/` — project đã giải nén; hoặc
- `/content/recsys-hust.zip` — notebook sẽ tự giải nén.

Project phải chứa đầy đủ `data/processed/mind_large/`.

## GPU

- **BPR** dùng PyTorch và sẽ tự dùng CUDA nếu Colab có GPU.
- **Content-based** và phần lớn **evaluation** dùng CPU (NumPy/SciPy), nên GPU không tăng tốc toàn bộ pipeline.
- Nếu có thể, hãy chọn **GPU runtime + High-RAM**; GPU giúp train BPR, RAM/CPU giúp phần còn lại.

File ZIP cuối cùng có đúng cấu trúc project. Khi tải về, chỉ cần giải nén trực tiếp vào thư mục `recsys-hust` và chọn ghi đè.


In [ ]:
# Cấu hình chung
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

PROJECT_DIR = Path('/content/recsys-hust')
PROJECT_ZIP = Path('/content/recsys-hust.zip')

# Tùy chọn backup file kết quả cuối lên Google Drive.
USE_GOOGLE_DRIVE_BACKUP = False
DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/recsys-hust-data-minimization')

print('PROJECT_DIR =', PROJECT_DIR)
print('PROJECT_ZIP =', PROJECT_ZIP)


In [ ]:
# Tự giải nén nếu project chưa có nhưng recsys-hust.zip đã được upload.
if not PROJECT_DIR.exists():
    if not PROJECT_ZIP.exists():
        raise FileNotFoundError(
            'Không thấy /content/recsys-hust hoặc /content/recsys-hust.zip. '
            'Hãy upload project trước khi chạy cell này.'
        )

    print('Đang giải nén project...')
    shutil.unpack_archive(str(PROJECT_ZIP), '/content')

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        'Sau khi giải nén vẫn không thấy /content/recsys-hust. '
        'Hãy kiểm tra thư mục gốc bên trong file ZIP.'
    )

os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())


In [ ]:
# Kiểm tra các file/dataset bắt buộc.
required_paths = [
    Path('requirements.txt'),
    Path('src/evaluation/data_minimization.py'),
    Path('configs/experiment/data_minimization.yaml'),
    Path('data/processed/mind_large/train/pointwise'),
    Path('data/processed/mind_large/train/positive_interactions'),
    Path('data/processed/mind_large/dev/impressions'),
    Path('data/processed/mind_large/mappings/items'),
]

missing_paths = [str(path) for path in required_paths if not path.exists()]

if missing_paths:
    print('Thiếu các đường dẫn sau:')
    for path in missing_paths:
        print(' -', path)
    raise FileNotFoundError('Project hoặc dữ liệu chưa đầy đủ.')

print('OK: Đã tìm thấy toàn bộ file/dataset cần thiết.')


In [ ]:
# Cài dependencies.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '--quiet'],
    check=True,
)

print('Cài requirements xong.')


In [ ]:
# Kiểm tra CPU/GPU và xác nhận BPR sẽ chọn device nào.
import platform
import torch
import pyarrow
import pandas as pd
import numpy as np

print('Python       :', sys.version.split()[0])
print('Platform     :', platform.platform())
print('PyTorch      :', torch.__version__)
print('pyarrow      :', pyarrow.__version__)
print('pandas       :', pd.__version__)
print('numpy        :', np.__version__)
print('CUDA usable  :', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU          :', torch.cuda.get_device_name(0))
    print('BPR device   : cuda')
else:
    print('BPR device   : cpu')
    print('Gợi ý: Runtime > Change runtime type > chọn GPU nếu muốn tăng tốc phần train BPR.')


## A. Chuẩn bị retention datasets

Tạo lại `1d / 3d / 5d / all` bằng cùng cách ghi Parquet để so sánh storage công bằng.

Bước này **chưa train model**.


In [ ]:
# Prepare-only với overwrite để bốn window đồng nhất.
command = [
    sys.executable,
    '-m',
    'src.evaluation.data_minimization',
    '--prepare-only',
    '--overwrite-data',
]

print('Running:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
# Hiển thị thống kê retention/storage.
stats_path = Path('reports/data_minimization/dataset_stats.csv')

if not stats_path.exists():
    raise FileNotFoundError(f'Không thấy {stats_path}')

stats = pd.read_csv(stats_path)
display(stats)


## B. Chạy full experiment

Pipeline chạy tuần tự:

1. `1d → BPR → Content`
2. `3d → BPR → Content`
3. `5d → BPR → Content`
4. `all → BPR → Content`

BPR sẽ tự dùng GPU nếu cell kiểm tra phía trên báo `BPR device: cuda`.

Content-based và evaluation vẫn chủ yếu dùng CPU.


In [ ]:
# Chạy toàn bộ Data Minimization experiment.
command = [
    sys.executable,
    '-u',
    '-m',
    'src.evaluation.data_minimization',
]

print('Running:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
# Hiển thị summary cuối.
summary_path = Path('reports/data_minimization/summary.csv')

if not summary_path.exists():
    raise FileNotFoundError(f'Không thấy {summary_path}')

summary = pd.read_csv(summary_path)
display(summary)


In [ ]:
# Liệt kê các biểu đồ đã sinh.
figures_dir = Path('reports/data_minimization/figures')

if figures_dir.exists():
    for figure_path in sorted(figures_dir.glob('*.png')):
        print(figure_path)
else:
    print('Chưa thấy thư mục figures.')


## C. Tạo ZIP đúng cấu trúc project

ZIP cuối sẽ có trực tiếp:

```text
reports/data_minimization/...
artifacts/data_minimization/...
```

Sau khi tải về Windows:

1. Mở `C:\Users\Admin\Downloads\recsys-hust`.
2. Giải nén ZIP vào chính thư mục này.
3. Chọn **Replace/Overwrite** nếu Windows hỏi.

Không cần copy thủ công từng thư mục.


In [ ]:
# Nén kết quả với đường dẫn tương đối đúng theo project.
output_zip = Path('/content/data_minimization_project_patch.zip')

if output_zip.exists():
    output_zip.unlink()

sources = [
    Path('reports/data_minimization'),
    Path('artifacts/data_minimization'),
]

with zipfile.ZipFile(output_zip, mode='w', compression=zipfile.ZIP_DEFLATED) as archive:
    for source_dir in sources:
        if not source_dir.exists():
            print('Bỏ qua vì không tồn tại:', source_dir)
            continue

        for file_path in source_dir.rglob('*'):
            if not file_path.is_file():
                continue

            # Giữ đúng path: reports/data_minimization/... hoặc artifacts/data_minimization/...
            archive.write(file_path, arcname=file_path.as_posix())

print('Đã tạo:', output_zip)
print('Dung lượng:', round(output_zip.stat().st_size / (1024 ** 2), 2), 'MB')


In [ ]:
# Kiểm tra nhanh cấu trúc bên trong ZIP trước khi tải.
with zipfile.ZipFile('/content/data_minimization_project_patch.zip', 'r') as archive:
    names = archive.namelist()

print('Số file trong ZIP:', len(names))
print('\n10 file đầu:')
for name in names[:10]:
    print(' -', name)

if not any(name.startswith('reports/data_minimization/') for name in names):
    raise RuntimeError('ZIP thiếu reports/data_minimization/.')

if not any(name.startswith('artifacts/data_minimization/') for name in names):
    raise RuntimeError('ZIP thiếu artifacts/data_minimization/.')

print('\nOK: ZIP có đúng cấu trúc để giải nén vào project.')


In [ ]:
# Tải ZIP về máy.
from google.colab import files

files.download('/content/data_minimization_project_patch.zip')


## D. Tùy chọn backup lên Google Drive

Nếu muốn lưu thêm bản ZIP trên Google Drive, đổi `USE_GOOGLE_DRIVE_BACKUP = True` ở cell đầu rồi chạy cell dưới.


In [ ]:
# Backup ZIP + summary lên Google Drive nếu được bật.
if USE_GOOGLE_DRIVE_BACKUP:
    from google.colab import drive

    drive.mount('/content/drive')
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

    shutil.copy2(
        '/content/data_minimization_project_patch.zip',
        DRIVE_BACKUP_DIR / 'data_minimization_project_patch.zip',
    )

    shutil.copy2(
        'reports/data_minimization/summary.csv',
        DRIVE_BACKUP_DIR / 'summary.csv',
    )

    print('Backup xong:', DRIVE_BACKUP_DIR)
else:
    print('Google Drive backup đang tắt.')
